In [33]:
import os
import json
import pandas as pd
import numpy as np
import geopandas as gpd
import subprocess
from tqdm import tqdm

In [34]:
R2D_E1_tempdir = "/resnick/groups/enceladus/jyzhao/cybershake_ngmm_prediction/risk_calculation/Exploring_code/tmp.SimCenter_E1"
R2D_E1_config_fp = os.path.join(R2D_E1_tempdir, "inputRWHALE.json")
with open(R2D_E1_config_fp, 'r') as f:
    R2D_E1_config = json.load(f)
pypsa_config = R2D_E1_config.copy()
num_samples = 1000

In [35]:
eq_scenario_id = "bssc2014ssanandreasnsbssbbgc_m7p64_se"
expansion_scenario = "conservative"

In [36]:
# asset_bau_fp = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/eq_handoff_fullcats/bau/disagg/assets_all.csv"
# asset_bau_df = pd.read_csv(asset_bau_fp)
# bau_substations = asset_bau_df[asset_bau_df['asset_type'] != 'storage_unit']
# bau_substations = bau_substations[bau_substations['new_mw'] == 0]
# bau_substations_loc = bau_substations[['lon', 'lat']].apply(
#     lambda x: (x['lon'], x['lat']),
#     axis=1
# )

In [37]:
eq_scenario_dir = f"/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/BSSC2014_scenarios/{eq_scenario_id}"
asset_gm_fp = os.path.join(eq_scenario_dir, expansion_scenario, "combined_assets_with_pga.csv")
asset_gm_df = pd.read_csv(asset_gm_fp)
asset_gm_df.columns


Index(['lon', 'lat', 'v_nom_kv', 'bus', 'asset_type', 'new_mw', 'asset_id',
       'id', 'site_pga'],
      dtype='str')

In [38]:
asset_gm_df['asset_type'].value_counts()

asset_type
Substation      8899
generator       3016
storage_unit      15
Name: count, dtype: int64

In [39]:
# Create pelicun readable inventory csv files
asset_gm_df['id'] = asset_gm_df['id'].astype(str)
# Pelicun assume the first column is 'id'
id_col = asset_gm_df.pop('id')
asset_gm_df.insert(0, 'id', id_col)


substation_df = asset_gm_df[asset_gm_df['asset_type'] == 'Substation']
# Add "type" as "Substation" # one substation per bus
substation_df['type'] = "Substation"
# Add "Voltage" as v_nom
substation_df['Voltage'] = substation_df['v_nom_kv']
# Set "Anchored" as true
substation_df['Anchored'] = "true"
number_of_substations = len(substation_df)

# generation
generation_df = asset_gm_df[asset_gm_df['asset_type'] == 'generator']
generation_df = generation_df[generation_df['new_mw'] > 0]
number_of_generators = len(generation_df)

# Add "type" as "Generation"
generation_df['type'] = "Generation"
# Add "Output" as v_nom
generation_df['Output'] = generation_df['new_mw'].apply(lambda x: f"{x*1e6} w")
# Set "Anchored" as true
generation_df['Anchored'] = "true"

# storage unit
storage_df = asset_gm_df[asset_gm_df['asset_type'] == 'storage_unit']
# Add "type" as "Generation" (Use the same fragility as small generation plants)
storage_df['type'] = "Generation"
# Add "Capacity" as v_nom
storage_df['Output'] = f"{50*1e6} w"
# Set "Anchored" as true
storage_df['Anchored'] = "true"

asset_gm_df = pd.concat([substation_df, generation_df, storage_df], ignore_index=True)

display(asset_gm_df.head(1))
print(number_of_substations, number_of_generators, len(storage_df))

,id,lon,lat,v_nom_kv,bus,asset_type,new_mw,asset_id,site_pga,type,Voltage,Anchored,Output
0,0,-122.242916,37.772436,115.0,CATS_1,Substation,NaN,NaN,0.0,Substation,115.0,true,NaN


8899 708 15


In [40]:
asset_info_csv_dir = os.path.join(eq_scenario_dir, expansion_scenario)
asset_info_csv_fn = "asset_info_pelicun.csv"
asset_gm_df.to_csv(os.path.join(asset_info_csv_dir, 
    asset_info_csv_fn),
                      index=False)

In [41]:
# Modify pypsa_config related to inventory files as needed
pypsa_config['Applications']['Assets']['Buildings'][
        'ApplicationData'
    ]['pathToSource'] = asset_info_csv_dir
pypsa_config['Applications']['Assets']['Buildings'][
    'ApplicationData'
    ]['assetSourceFile'] = asset_info_csv_fn
pypsa_config['Applications']['Assets']['Buildings'][
            'ApplicationData'
        ].pop('filter', None)
# Modify pypsa_config dl method
pypsa_config['Applications']['DL']['Buildings']['ApplicationData'][
        'DL_Method'
    ] = 'Hazus Earthquake - Electric Power'
# Modify pypsa_config variables to only save Damage
pypsa_config['SaveVariables'] = ['Demand','Damage']
# Modify number of damage state samples
pypsa_config['Applications']['DL']['Buildings'][
        'ApplicationData'
    ]['Realizations'] = num_samples


In [42]:
# Create the gm sample file
pga_samples = pd.DataFrame(
    # Manually make it two identical realization to avoid pelicun error
    # np.vstack([asset_gm_df['site_pga'].values.reshape(1, -1)]*2),
    asset_gm_df['site_pga'].values.reshape(1, -1),
    columns=asset_gm_df['id'].values
)
new_columns = pd.MultiIndex.from_product([pga_samples.columns, ['PGA']])
pga_samples.columns = new_columns
display(pga_samples.head(3))

pga_samples.to_csv(
    os.path.join(eq_scenario_dir, expansion_scenario, "pga_samples_pelicun.csv"),
    index=False,
)
# Replace the RegionalMapping application name to UserDefinedDemandSample
pypsa_config['Applications']['RegionalMapping']['Buildings'][
        'Application'] = 'UserDefinedDemandSample'
pypsa_config['RegionalEvent']['eventFilePath'] = asset_info_csv_dir
pypsa_config['RegionalEvent']['eventFile'] = f'pga_samples_pelicun.csv'

pypsa_config_fp = os.path.join(
    asset_info_csv_dir, f"pypsa_config_pelicun.json"
)
with open(pypsa_config_fp, 'w') as f:
        json.dump(pypsa_config, f, indent=4)

,0,1,2,3,4,5,6,7,8,9,...,11920,11921,11922,11923,11924,11925,11926,11927,11928,11929
,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,...,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA,PGA
0,0.0,0.0,0.0,0.44165,0.173468,0.234926,0.276983,0.595979,0.628008,0.204852,...,0.0,0.0,0.010031,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
pga_samples.max().max()

np.float64(0.653852310180664)

In [44]:
# command to run pelicun
pelicun_path = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/python_libs/pelicun_cli.py"
python_path = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/.venv_pelicun/bin/python"

run_dir = asset_info_csv_dir
curdir = os.getcwd()
os.chdir(run_dir)
command = f"{python_path} -u {pelicun_path} regional_sim {pypsa_config_fp} -n 1"
try:
    subprocess.run(command, shell=True, check=True)
except subprocess.CalledProcessError as e:
    print(f"Command failed with error: {e}")

os.chdir(curdir)

/resnick/groups/enceladus/jyzhao/CAPowerResilience/python_libs/pelicun/pelicun/__init__.py:61: PelicunWarning: DLML data update available. Current version: v2.1.0, Latest version: v3.2. Update with: /resnick/groups/enceladus/jyzhao/CAPowerResilience/.venv_pelicun/bin/python -m pelicun.cli dlml update
  check_dlml_data()


DLML data update available. Current version: v2.1.0, Latest version: v3.2. Update with: /resnick/groups/enceladus/jyzhao/CAPowerResilience/.venv_pelicun/bin/python -m pelicun.cli dlml update
[00:00:00] 2 Building Inventory


Processing building chunks: 100%|██████████| 10/10 [03:41<00:00, 22.19s/it]


[00:03:43] 7 Save results


In [45]:
asset_gm_df['id'] = asset_gm_df['id'].astype(int)
damage_id_to_asset_id = asset_gm_df.set_index('id')['asset_id'].to_dict()
damage_id_to_bus_id = asset_gm_df.set_index('id')['bus'].to_dict()

In [46]:
damage_df_all = pd.read_csv(os.path.join(
    asset_info_csv_dir, "damage.csv"
))

substation_damage_df = damage_df_all[damage_df_all['id'].isin(
    asset_gm_df.iloc[:number_of_substations]['id']
)].copy()
substation_damage_df['bus'] = substation_damage_df['id'].map(damage_id_to_bus_id)

generator_damage_df = damage_df_all[damage_df_all['id'].isin(
    asset_gm_df.iloc[number_of_substations:number_of_substations+number_of_generators]['id']
)].copy()
generator_damage_df['asset_id'] = generator_damage_df['id'].map(damage_id_to_asset_id)

storage_damage_df = damage_df_all[damage_df_all['id'].isin(
    asset_gm_df.iloc[number_of_substations+number_of_generators:]['id']
)].copy()
storage_damage_df['asset_id'] = storage_damage_df['id'].map(damage_id_to_asset_id)
print(substation_damage_df.shape, generator_damage_df.shape, storage_damage_df.shape)

substation_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_substation.csv"), index=False)
generator_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_generator.csv"), index=False)
storage_damage_df.to_csv(os.path.join(asset_info_csv_dir, "damage_storage.csv"), index=False)

(8899, 1002) (708, 1002) (15, 1002)


In [47]:
# Get remaining functional capacity according to Table 8-28 in HAZUS 6.1
substation_damage_to_remaining_functional_capacity = {
    0: 1.0,
    1: 0.5,
    2: 0.09,
    3: 0.04,
    4: 0.03
}
generator_damage_to_remaining_functional_capacity = {
    0: 1.0,
    1: 1.0,
    2: 0.24,
    3: 0.16,
    4: 0.02
}
storage_damage_to_remaining_functional_capacity = generator_damage_to_remaining_functional_capacity

for i in tqdm(range(num_samples), total=num_samples):
    col_name = str(i)
    substation_damage_df[col_name] = substation_damage_df[col_name].map(
        substation_damage_to_remaining_functional_capacity)
    generator_damage_df[col_name] = generator_damage_df[col_name].map(
        generator_damage_to_remaining_functional_capacity)
    storage_damage_df[col_name] = storage_damage_df[col_name].map(
        storage_damage_to_remaining_functional_capacity)

100%|██████████| 1000/1000 [00:01<00:00, 625.96it/s]


In [48]:
substation_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_substation.csv"), index=False)
generator_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_generator.csv"), index=False)
storage_damage_df.to_csv(os.path.join(asset_info_csv_dir, "capacity_storage.csv"), index=False)